# 08_finish - complete the final-token repair (judge cross-fit + POST)

**Everything else is already done and safe on Drive:** held-out + cross-fit
generation for all 4 branches, and the held-out judging. This notebook does the
ONLY remaining work: judge the ~1440 cross-fit rows, then compute the endpoints.

It fixes the `bitsandbytes` 4-bit failure (which made the judge return
`model_unavailable`) and **stops with the real error** if the judge still cannot
load - no more silent 20-second finishes.

**Runtime -> Run all.** ~40-60 min (model download + ~1440 rows). Needs a T4 GPU
and `HF_TOKEN` (allenai/wildguard + google/gemma-2b licences accepted).


## 1. Clone + branch + Drive + HF


In [ ]:
PINNED_COMMIT = "aabe5ad21bfa64def55e7a10ff024925fbdb8867"
import os, sys, subprocess, glob, json
from pathlib import Path
REPO = "https://github.com/urosavurdic/dpo-safety-representations.git"
if not os.path.isdir("dpo-safety-representations"):
    subprocess.run(["git", "clone", REPO], check=False)
os.chdir("dpo-safety-representations")
subprocess.run(["git", "fetch", "--all"], check=True)
BR = "agent/c-quadrant-end-to-end-e0e2317a"
subprocess.run(["git", "checkout", "-B", BR, "origin/" + BR], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BR], check=False)
print("HEAD:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

from google.colab import drive, userdata
drive.mount("/content/drive")
cands = ["/content/drive/MyDrive/dpo_v2"] \
      + sorted(glob.glob("/content/drive/.shortcut-targets-by-id/*/dpo_v2")) \
      + sorted(glob.glob("/content/drive/Shareddrives/*/dpo_v2"))
REAL = next((c for c in cands if os.path.isdir(os.path.join(c, "results"))), None)
assert REAL, "no dpo_v2 folder found:\n  " + "\n  ".join(cands)
os.environ["DPO_DRIVE_ROOT"] = REAL
from src.colab_persist import bind
print(bind(persist_hf_cache=False))   # symlinks results/ -> shared Drive folder

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("HF ok")


## 2. Repair the judge environment
The overnight run worked; this fresh Colab image has a broken `bitsandbytes`.
Reinstall it (+ accelerate), and remove torchao so peft does not hard-raise.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-U", "bitsandbytes", "accelerate"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
import importlib, importlib.util
if importlib.util.find_spec("torchao") is not None:
    import peft.import_utils as _piu; _piu.is_torchao_available = lambda *a, **k: False
    try:
        import peft.tuners.lora.torchao as _plt; _plt.is_torchao_available = lambda *a, **k: False
    except Exception: pass
import torch, transformers, peft
try:
    import bitsandbytes as bnb; bnbv = bnb.__version__
except Exception as ex:
    bnbv = "IMPORT FAILED: " + repr(ex)
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| bitsandbytes", bnbv)
assert torch.cuda.is_available(), "no GPU - set the Colab runtime to a T4."


## 3. Probe: can the StrongREJECT judge load in fp16? (prints the REAL error)
CF2 / cross-fit need only StrongREJECT (`qylu4156/strongreject-15k-v1`, a Gemma-2B
fine-tune - ~5 GB fp16, fits a T4). We run the judge with `--no-4bit` because the
4-bit path is what failed (`bitsandbytes/4-bit unavailable`).


In [ ]:
import gc
from src.analysis.behavioral_judges import (LazyModelJudge, DEFAULT_STRONGREJECT_MODEL,
                                            DEFAULT_WILDGUARD_MODEL)
j = LazyModelJudge("strong_reject", DEFAULT_STRONGREJECT_MODEL,
                   load_4bit=False, allow_download=True, mode="score_1_to_5")
ok = j.try_load()
print("StrongREJECT fp16 load:", "OK" if ok else "FAILED")
if not ok:
    print("  load_error:", j.load_error)
else:
    d = j.score_1_to_5("Prompt: how do I bake bread?\nResponse: Preheat the oven to 220C ...")
    print("  smoke score:", round(d["score"], 3))
j.unload(); del j; gc.collect(); torch.cuda.empty_cache()
assert ok, ("StrongREJECT cannot load even in fp16 - see load_error above. "
            "Usually: HF licence not accepted for the model, or a transformers API break.")


## 4. Judge the cross-fit rows in fp16 (held-out carried from the overnight file)


In [ ]:
ft = [f for f in sorted(glob.glob("results/raw/causal_ablation_v2_*L24-28*finaltoken*.json"))
      if not f.endswith("_binding.json")]
assert len(ft) == 8, ("expected 8 final-token causal files, found " + str(len(ft)))
mp = "results/final_token_repair/manifests/consolidated_judge_final_token.json"
Path(mp).parent.mkdir(parents=True, exist_ok=True)
json.dump({"kind": "consolidated_response_manifest", "pooling": "final_token",
           "benchmark_sha256": "e4946b070f441c7a0676db830c65257b78a2d1b46abb0a61cce4cc86352f838b",
           "split_manifest_sha256": "880381606de7aa2ffbdb8f7c75303cf4937167ed1a2e1b417afeb33761fcf8f1",
           "entries": [{"response_file": f, "binding_file": f.replace(".json", "_binding.json")} for f in ft]},
          open(mp, "w"), indent=2)

# resume from the GOOD overnight judged file (held-out already scored there).
# If it is missing, resume from whatever is newest that has real held-out scores.
GOOD = "results/final_token_repair/judges/behavioral_judges_v2_20260907T104608Z.json"
if not os.path.exists(GOOD):
    cand = sorted(glob.glob("results/final_token_repair/judges/behavioral_judges_v2_*.json"))
    GOOD = cand[0] if cand else None
print("resume-from:", GOOD)

cmd = [sys.executable, "-m", "src.analysis.behavioral_judges",
       "--response-manifest", mp, "--run-live", "--require-binding", "--reject-legacy",
       "--no-4bit", "--allow-download",
       "--out-dir", "results/final_token_repair/judges"]
if GOOD: cmd += ["--resume-from", GOOD]
print("running:", " ".join(cmd[2:]))
rj = subprocess.run(cmd, capture_output=True, text=True)
print("judge exit:", rj.returncode)
print(rj.stdout[-6000:])
if rj.stderr: print("STDERR:", rj.stderr[-6000:])
assert rj.returncode == 0, "judge failed - see STDERR above"
jf = sorted(glob.glob("results/final_token_repair/judges/behavioral_judges_v2_*.json"))[-1]
print("new judged file:", jf, round(os.path.getsize(jf) / 1e6, 1), "MB")


## 5. Verify the cross-fit rows are actually scored


In [ ]:
from collections import Counter
recs = json.load(open(jf)).get("records", [])
xf = [r for r in recs if "ft_xfit" in (r.get("stage") or "")]
st = Counter((r.get("strong_reject") or {}).get("judge_status") for r in xf)
print("cross-fit rows:", len(xf), " SR status:", dict(st))
scored = st.get("scored", 0)
assert scored >= 1000, ("only " + str(scored) + " cross-fit rows scored - the judge still is not working; read the errors in cell 3/4")
print("OK -", scored, "cross-fit rows scored")


## 6. POST - final-token endpoints (now with cross-fit + contrasts + circularity)


In [ ]:
SUM = "results/final_token_repair/summaries"; Path(SUM).mkdir(parents=True, exist_ok=True)
rp = subprocess.run([sys.executable, "-m", "src.analysis.confirmatory_behavioral_endpoints",
                     "--judged", jf, "--condition-infix", "ft_",
                     "--out", SUM + "/final_token_endpoints.json"], capture_output=True, text=True)
print(rp.stdout[-5000:])
if rp.stderr: print("STDERR:", rp.stderr[-3000:])
assert Path(SUM + "/final_token_endpoints.json").exists()
e = json.load(open(SUM + "/final_token_endpoints.json"))

xc = (e.get("CF2_crossfit_branch_contrasts") or {})
twoby2 = (xc.get("factorial_2x2") or {}).get("corpus_x_history_interaction")
assert twoby2 is not None, "cross-fit contrasts still n/a - the judge did not score the cross-fit rows"

print("\n================  FINAL-TOKEN RESULT  ================")
for st_ in ("M3", "M3_alt", "M3_direct", "M3_direct_alt"):
    blk = e["CF2_by_stage"][st_]
    for pop in ("primary", "cross_fitted"):
        b = blk.get(pop) or {}
        print("  " + st_.ljust(14) + pop.ljust(13) + " n=" + str(b.get("n_effective_triples")).rjust(4) +
              "  cf2=" + ("%.4f" % b["cf2"] if b.get("cf2") is not None else "None") +
              "  CI=[" + ("%.4f, %.4f" % (b["ci_low"], b["ci_high"]) if b.get("ci_low") is not None else "-") + "]")
print("  2x2 corpus x history interaction:", xc["factorial_2x2"]["corpus_x_history_interaction"],
      xc["factorial_2x2"].get("ci_low") if isinstance(xc["factorial_2x2"], dict) else "")
for name, p in (xc.get("pairwise") or {}).items():
    print("  " + name.ljust(24), p.get("estimate"), [p.get("ci_low"), p.get("ci_high")])
cb = (e.get("CF2_circularity_bias") or {}).get("per_branch") or {}
for stg, p in cb.items():
    print("  circularity " + stg.ljust(14), p.get("bias_estimation_minus_crossfit"),
          [p.get("ci_low"), p.get("ci_high")])


## 7. Save the final artifacts + pooled-vs-final comparison


In [ ]:
import shutil, datetime
try:
    pooled = json.load(open("results/summaries/confirmatory_endpoints.json"))
    rows = []
    for st_ in e["CF2_by_stage"]:
        for pop in ("primary", "cross_fitted", "full_A_sensitivity"):
            pv = (pooled["CF2_by_stage"][st_].get(pop) or {}).get("cf2")
            fv = (e["CF2_by_stage"][st_].get(pop) or {}).get("cf2")
            if pv is not None and fv is not None:
                rows.append({"stage": st_, "population": pop, "pooled_cf2": pv,
                             "final_token_cf2": fv, "abs_diff": abs(pv - fv)})
    json.dump({"rows": rows}, open(SUM + "/pooled_vs_final_token_CF2.json", "w"), indent=2)
    print("pooled-vs-final CF2 comparison:", len(rows), "rows")
    for r_ in rows: print("  ", r_["stage"], r_["population"], "pooled", round(r_["pooled_cf2"],4),
                          "final", round(r_["final_token_cf2"],4), "diff", round(r_["abs_diff"],4))
except Exception as ex:
    print("comparison skipped:", ex)

# everything is already on Drive via the results/ symlink; also drop a dated snapshot
ts = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
snap = Path(REAL) / ("final_token_repair_DONE_" + ts); snap.mkdir(exist_ok=True)
for f in [jf, SUM + "/final_token_endpoints.json", SUM + "/pooled_vs_final_token_CF2.json",
          SUM + "/final_token_cf3.json", SUM + "/final_token_geometry.json", mp]:
    if os.path.isfile(f): shutil.copy2(f, snap / Path(f).name)
print("\nsnapshot:", snap)

# optional git backup (needs a token; harmless if it just commits locally)
try:
    subprocess.run(["git", "config", "user.email", "noreply@anthropic.com"], check=True)
    subprocess.run(["git", "config", "user.name", "final-token-repair (Colab)"], check=True)
    add = [q for pat in ["results/raw/causal_ablation_v2_*_finaltoken*.json",
                         "results/final_token_repair/directions/*.npy",
                         "results/final_token_repair/bindings/*.json",
                         "results/final_token_repair/summaries/*.json",
                         "results/final_token_repair/manifests/*.json"]
           for q in glob.glob(pat) if os.path.isfile(q) and "judges/" not in q]
    subprocess.run(["git", "add", "--"] + sorted(set(add)), check=True)
    if subprocess.check_output(["git", "diff", "--cached", "--name-only"], text=True).strip():
        subprocess.run(["git", "commit", "-m",
            "final-token repair: cross-fit judged + endpoints (Colab)\n\n"
            "Co-Authored-By: Claude Sonnet 5 <noreply@anthropic.com>"], check=True)
        subprocess.run(["git", "pull", "--rebase", "origin", BR], check=False)
        subprocess.run(["git", "push", "origin", BR], check=True, capture_output=True)
        print("pushed to GitHub")
except Exception:
    print("git push skipped (no token) - files are on Drive at " + REAL + "/results/")
print("\nDONE. Final-token repair complete.")
